# RAG with Two Chunking Strategies

*   List item
*   List item


simple RAG system comparing `RecursiveCharacterTextSplitter` vs `MarkdownHeaderTextSplitter`, served via a Gradio UI.





#Install packages

In [1]:
!pip install -q \
  langchain-chroma \
  langchain-huggingface \
  langchain-text-splitters \
  sentence-transformers \
  chromadb \
  gradio \
  wikipedia

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages t

#1.Imports


In [2]:
import torch
import wikipedia
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio as gr

## 2. Build the corpus

A handful of Wikipedia articles i am intrested in. Pick topics you actually want to query focused content makes the retrieval comparison more interesting.

In [3]:
topics = [
    "Information theory",
    "Claude Shannon",
    "Machine learning",
    "Saudi Arabia",
    "Quantum computing",
    "Transformer (deep learning architecture)",
]

texts = []
for t in topics:
    try:
        page = wikipedia.page(t, auto_suggest=False)
        texts.append({"title": page.title, "content": page.content})
        print(f"OK   {page.title}  ({len(page.content)} chars)")
    except Exception as e:
        print(f"SKIP {t}: {e}")

markdown_texts = [f"# {t['title']}\n\n{t['content']}" for t in texts]
print(f"\nTotal documents: {len(markdown_texts)}")

OK   Information theory  (57748 chars)
OK   Claude Shannon  (33683 chars)
OK   Machine learning  (58652 chars)
OK   Saudi Arabia  (97449 chars)
OK   Quantum computing  (62142 chars)
OK   Transformer (deep learning)  (113931 chars)

Total documents: 6


## 3. Chunking method 1 — Recursive character splitter

Splits on a hierarchy of separators (paragraphs → lines → words → chars). Structure-agnostic — doesn't know about markdown headers, just tries to keep chunks near the target size.
medium articule that explains it pretty well.

http://medium.com/@nihalgupta65/understanding-recursive-chunking-in-rag-how-it-works-when-it-fails-and-how-to-think-about-it-489ce043b235

In [4]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
docs_recursive = recursive_splitter.create_documents(markdown_texts)

print(f"Recursive chunker produced {len(docs_recursive)} chunks")
print("\nExample chunk:")
print(docs_recursive[0].page_content[:300])

Recursive chunker produced 1287 chunks

Example chunk:
# Information theory


In [8]:
print(docs_recursive[-1].page_content[:300])

== Notes ==


== References ==


== Further reading ==


## 4. Chunking method 2 Markdown header splitter

Splits at `#`, `##`, `###` boundaries and stores the header path as metadata. More structure aware and respects the document outline.

A secondary recursive splitter caps large sections at 500 chars so chunks stay comparable in size to method 1.

In [9]:
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
)

docs_markdown = []
for text in markdown_texts:
    docs_markdown.extend(md_splitter.split_text(text))


secondary = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs_markdown = secondary.split_documents(docs_markdown)

print(f"Markdown chunker produced {len(docs_markdown)} chunks")
print("\nExample chunk metadata:", docs_markdown[0].metadata)
print("\nExample chunk:")
print(docs_markdown[0].page_content[:300])

Markdown chunker produced 985 chunks

Example chunk metadata: {'Header 1': 'Information theory'}

Example chunk:
Information theory is the mathematical study of the quantification, storage, and communication of a particular type of mathematically defined information. The field was established and formalized by Claude Shannon in the 1940s, though early contributions were made in the 1920s through the works of H


## 5. Embed and create two vector stores



In [10]:
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db_recursive = Chroma.from_documents(
    docs_recursive,
    embedding_function,
    collection_name="recursive",
    persist_directory="./chroma_recursive",
)

db_markdown = Chroma.from_documents(
    docs_markdown,
    embedding_function,
    collection_name="markdown",
    persist_directory="./chroma_markdown",
)

print("Both vector stores built.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Both vector stores built.


# 6. Load the language model

Qwen2.5-1.5B-Instruct: small (~3GB in fp16), ungated, strong at instruction following..



In [11]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print(f"Loaded {MODEL_ID} on {model.device}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct on cuda:0


## 7. The RAG function

Retrieves top-k chunks, formats them with the user question via the model's chat template, generates an answer.

In [13]:
SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer the user's question using only the provided context. "
    "If the context doesn't contain the answer, say so clearly."
)

def query_rag(question: str, db, k: int = 4):

    docs = db.similarity_search(question, k=k)
    context = "\n\n".join(doc.page_content for doc in docs)


    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )
    return answer.strip(), docs



answer, retrieved = query_rag("Entropy?", db_recursive)
print("ANSWER:")
print(answer)
print("\nRETRIEVED CHUNKS:")
for i, d in enumerate(retrieved, 1):
    print(f"\n--- chunk {i} ---")
    print(d.page_content[:200], "...")

ANSWER:
Entropy is a measure of the amount of uncertainty or randomness associated with the outcomes of a discrete random variable. It quantifies how much information is needed on average to describe the possible values of the variable. Higher entropy indicates that the outcomes are less predictable, while lower entropy suggests more certainty.

RETRIEVED CHUNKS:

--- chunk 1 ---
Much of the mathematics behind information theory with events of different probabilities were developed for the field of thermodynamics by Ludwig Boltzmann and J. Willard Gibbs. Connections between in ...

--- chunk 2 ---
(
        1
        
          /
        
        2
        )
      
    
    {\displaystyle -\log _{2}(1/2)}
  
 = 1 bit of information.
A key concept in information theory is entropy. In Shannon's f ...

--- chunk 3 ---
Intuitively, the entropy 
  
    
      
        H
        (
        X
        )
      
    
    {\displaystyle H(X)}
  
 of a discrete random variable X is a measure of the

## 8. Gradio interface

Pick a chunking strategy, ask a question, see the answer and the retrieved chunks side by side. `share=True` produces a public link (~72h) you can send to your tutor.

In [14]:
def answer_with_ui(question, chunker):
    db = db_recursive if chunker == "Recursive" else db_markdown
    answer, retrieved = query_rag(question, db)
    chunks_preview = "\n\n---\n\n".join(
        [f"[chunk {i+1}]\n{d.page_content}" for i, d in enumerate(retrieved)]
    )
    return answer, chunks_preview

demo = gr.Interface(
    fn=answer_with_ui,
    inputs=[
        gr.Textbox(label="Question", lines=2,
                   placeholder="Ask something about the corpus..."),
        gr.Radio(["Recursive", "Markdown Header"],
                 label="Chunking method", value="Recursive"),
    ],
    outputs=[
        gr.Textbox(label="Answer", lines=4),
        gr.Textbox(label="Retrieved chunks", lines=15),
    ],
    title="RAG: Recursive vs Markdown-Header Chunking",
    description="Compare retrieval quality between the two chunking strategies.",
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bff71986be2b7ae055.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
